#Dimension Table - Members

> ## 1. View Data First

In [0]:
from pyspark.sql import functions as F

In [0]:
members = spark.table("himalaya.silver.expeditions_members")
peaks = spark.table("himalaya.silver.expeditions_peaks")
deaths = spark.table("himalaya.silver.deaths")

In [0]:
deaths_with_peakid = deaths.join(peaks, deaths["mountain"] == peaks["peak_name"], "left") \
    .select(deaths["name"], peaks["peakid"])

> ##2. Drop Ingestion

  _not needed in gold_

In [0]:
members = members.drop("ingested_at")

>##2A. Drop death related
_deaths will be stored separately in dim deaths_

In [0]:
members = members.drop("death_date", "death_type", "death_height")

>##3. Deaths after checked with deaths

In [0]:
members = members.join(deaths_with_peakid, 
    (members["name"] == deaths_with_peakid["name"]) & 
    (members["peakid"] == deaths_with_peakid["peakid"]), "left") \
    .withColumn("death", F.when(deaths_with_peakid["name"].isNotNull(), True).otherwise(F.col("death"))) \
    .select(
        members["expid"], members["peakid"], members["year"], members["season"],
        members["name"], members["sex"], members["yob"], members["nationality"],
        members["leader"], members["hired"], members["sherpa"], members["success"],
        members["highest_point"], members["summit_date"], members["summit_term"],
        members["oxygen_used"], "death"
    )

>##4. Write to Gold

In [0]:
members.write.format("delta").mode("overwrite").saveAsTable("himalaya.gold.dim_members")
print("✅ Written to himalaya.gold.dim_members")

In [0]:
display(spark.table("himalaya.gold.dim_members").limit(5))